In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
def Matrices_LU(A): 
    
    n = len(A) # Tamaño de la matriz A
    
    # Se crean matrices L y U de ceros con el mismo tamaño que A
    L = np.zeros((n, n))
    U = np.zeros((n, n))
    
    # Algoritmo de la variante Doolittle
    for i in range(n):
        L[i, i] = 1
        for j in range(i, n):
                s = sum(L[i, k] * U[k, j] for k in range(i))
                U[i, j] = A[i, j] - s
            
        for j in range(i+1, n):
                s = sum(L[j, k] * U[k, i] for k in range(i))
                L[j, i] = (A[j, i] - s) / U[i, i]
            
    return (L,U)

In [ ]:
def Sust_Hacia_Delante(L,Vector): 
    # Sustitución hacia delante Ly = b
    n = len(L) # Tamaño de la matriz L
    y = np.zeros(n) 
    for i in range(n):
            y[i] = Vector[i] - sum(L[i, j] * y[j] for j in range(i))
    return y

In [ ]:
def Sust_Hacia_Atrás(U,y):
# Sustitución hacia atrás Ux = y
    n = len(U) # Tamaño de la matriz U
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (y[i] - sum(U[i, j] * x[j] for j in range(i+1, n))) / U[i, i]
    
    return x

In [ ]:
def Resolución_Sistema_LU(A, Vector):
    resultado1 = Matrices_LU(A)
    # Utiliza el resultado de la función Matrices_LU
    y = Sust_Hacia_Delante(resultado1[0], Vector)
    sol = Sust_Hacia_Atrás(resultado1[1], y)
    return sol

In [ ]:
def calcula_lista_Deriv(lista_x, lista_f):
    
    # Los puntos son x_0, ..., x_n
    n=len(lista_x)-1
    
    # Calculo los valores de h y de delta
    h = np.zeros(n)
    delta = np.zeros(n)
    
    for i in range(0, n):
        h[i]=lista_x[i+1]-lista_x[i]
        delta[i] = (lista_f[i+1]-lista_f[i]) / h[i]
        
    # Obtengo la matriz de coeficientes y de los respectivos términos independientes
    A = np.zeros((n+1,n+1))
    B = np.zeros(n+1)
    
    # Obtengo en primer lugar la fila 0 y la fila n (son casos distintos)
    # Fila 0
    A[0][0] = 2/h[0]
    A[0][1] = 1/h[0]
    B[0] = delta[0]/h[0]

    # Fila n (ultima)
    A[n][n-1] = 1/h[n-1]
    A[n][n] = 2/h[n-1]
    B[n] = delta[n-1]/h[n-1]
    
    # Obtengo el resto de filas, iterando sobre cada fila
    for i in range(1, n):
        A[i][i-1] = 1/h[i-1]
        A[i][i+1] = 1/h[i]
        A[i][i]   = 2*(A[i][i-1] + A[i][i+1])
        
        B[i] = delta[i]/h[i] + delta[i-1]/h[i-1]
    
    B=3*B
    
    # Resuelvo el sistema mediante un algoritmo basado factorizacion LU y sustitucion hacia atrás
    # y hacia delante
    lista_d = Resolución_Sistema_LU(A, B)
    
    return(lista_d)

In [ ]:
def calc_dif_div_Hermite (lista_x, lista_f, lista_d):
    n = len(lista_x) # Numero de valores de las abcisas
    
    dif_divididas = np.zeros([n,n])

    # La tabla se va completando por columnas
    # En primer lugar, completo la columna 0
    for i in range(n):
        dif_divididas[i][0]=lista_f[i]
    
    # Para las demas utilizo los siguientes bucles para rellenarlas
    for k in range(1,n):
        for i in range(0, n-k):
            if (lista_x[i+k] == lista_x[i]):
                # Como se pasan las abcisas duplicadas, se divide entre 2
                dif_divididas[i][k] = lista_d[int(i/2)]
            
            else:
                dif_divididas[i][k] = (dif_divididas[i+1][k-1] - dif_divididas[i][k-1]) / (lista_x[i+k] -lista_x[i])

    # Introducimos los valores de las abcisas a la izquierda de la matriz
    # De esta manera, podemos reproducir exactamente el metodo que utilizabamos en papel
    adicional = np.array(lista_x).reshape(len(lista_x), 1)
    for i in range(n):
        adicional[i]=lista_x[i]
        
    dif_divididas = np.hstack((adicional, dif_divididas))
    return dif_divididas

In [ ]:
# Calculador Lista Polinomios
def spline(lista_x, lista_y):
    lista_d = calcula_lista_Deriv(lista_x, lista_y)
    n = len(lista_x) - 1  # Ajustar la longitud de la lista
    Lista_Polinomios = []

    for i in range(n):
        x_inter_x = np.array([lista_x[i], lista_x[i], lista_x[i + 1], lista_x[i + 1]])  # Corregir nombres de variables
        f_inter_x = np.array([lista_y[i], lista_y[i], lista_y[i + 1], lista_y[i + 1]])  # Corregir nombres de variables
        d_inter_x = np.array([lista_d[i], lista_d[i + 1]])  # Corregir nombres de variables
        # Aplico el metodo hermite para calcular el polinomio que se interpola en dicho intervalo
        Lista_Polinomios.append(calc_dif_div_Hermite(x_inter_x, f_inter_x, d_inter_x))  # Añadir polinomio a la lista

    return Lista_Polinomios

In [ ]:
def evaluar_Polinomio_Horner_Hermite(tabla_dif_div_Hermite, x_evaluar):
    n = len(tabla_dif_div_Hermite) - 1  # Ajustar el indice para evitar el error de indice fuera de rango
    resultado = tabla_dif_div_Hermite[0][n+1]
    
    # Aplicar método de Horner
    for i in range(n - 1, -1, -1):  # Iterar desde el penultimo coeficiente hacia atras
        resultado = resultado * (x_evaluar - tabla_dif_div_Hermite[i][0]) + tabla_dif_div_Hermite[0][i+1]
        
    return resultado

In [ ]:
def Evalua_Intervalo_en_Polinomio(pol_x, array_abcisas, lista_x):
    n = len(array_abcisas)
    array_imagen = np.zeros(n)

    for i in range(n):
        # Con ese -1 me refiero al ultimo de todos
        if(array_abcisas[i]<=lista_x[-1] and array_abcisas[i] >= lista_x[0]):
            # Buscar intervalo en el que se encuentra array_abcisas[i]
            j = 0
       
            while j < len(lista_x) and j < len(pol_x) and array_abcisas[i] >= lista_x[j] :
                j += 1

            # Evaluar polinomio en el punto correspondiente al intervalo
            array_imagen[i] = evaluar_Polinomio_Horner_Hermite(pol_x[j-1], array_abcisas[i])
 
        else: 
            print("Algunos de dichos puntos se salen del intervalo en el que esta definido el spline")

    return array_imagen

In [ ]:
def Divide_Matrix_t(lista_x, lista_f):
    n = len(lista_x)
    matrizX = np.empty((n, 2))
    matrizY = np.empty((n, 2))

    for i in range(0,n):
        matrizX[i][1] = lista_x[i]
        matrizX[i][0] = i+1  # Coordenadas de x
        matrizY[i][0] = i+1  # Coordenadas de y
        matrizY[i][1] = lista_f[i]

    return matrizX, matrizY

In [ ]:
def plotspline(matriz_puntos, color):
    lista_x = matriz_puntos[:,0]
    lista_f = matriz_puntos[:,1]
    # Divide en dos matrix_puntos
    Array_Matrix_Separado = Divide_Matrix_t(lista_x, lista_f)

    # Hallo hermite y vectores de polinomios
    array_polinomios_X = spline(Array_Matrix_Separado[0][:,0], Array_Matrix_Separado[0][:,1])
    array_polinomios_Y = spline(Array_Matrix_Separado[1][:,0], Array_Matrix_Separado[1][:,1])

    # Inventar array x
    n = len(Array_Matrix_Separado[0])
    abcisas_X = np.linspace(Array_Matrix_Separado[0][0,0], Array_Matrix_Separado[0][n-1,0], 400)
    abcisas_Y = np.linspace(Array_Matrix_Separado[1][0][0], Array_Matrix_Separado[1][n-1][0], 400)

    # Hacer imagenes
    imagenes_X = Evalua_Intervalo_en_Polinomio(array_polinomios_X, abcisas_X,Array_Matrix_Separado[0][:,0])
    imagenes_Y = Evalua_Intervalo_en_Polinomio(array_polinomios_Y,abcisas_Y, Array_Matrix_Separado[1][:,0])

    # Dibujar en el plano
    plt.plot(imagenes_X, imagenes_Y,color)

In [ ]:
def Dibujar_Completo(array_matriz_puntos, colores, color_fondo='k'):
    # Crear figura y ejes
    fig, ax = plt.subplots()
    
    # Establecer color de fondo
    ax.set_facecolor(color_fondo)
    
    # Dibujar cada conjunto de puntos
    for i in range(len(array_matriz_puntos)):
        plotspline(array_matriz_puntos[i], colores[i])
        
    ax.set_title('MÉTODOS NUMÉRICOS')
    # Mostrar la gráfica
    plt.show()



In [ ]:
array_matriz_puntos = []
colores_cada_spline = []

#Spline_1
matriz_Puntos_1 = np.array([[0.18, 4.97], [1, 4.94], [1.69, 4.27], [0.74, 4.39], [0.18, 4.97]])
colores_1 = 'forestgreen'
array_matriz_puntos.append(matriz_Puntos_1)
colores_cada_spline.append(colores_1)

#Spline_2
matriz_Puntos_2 = np.array([[2.23, 5.79], [2.43, 5.2], [2.25, 4.61], [1.81, 4.3],[1.78, 4.4],[1.74,4.52],[1.8,5.2],[2.23, 5.79]])
colores_2 = 'forestgreen'
array_matriz_puntos.append(matriz_Puntos_2)
colores_cada_spline.append(colores_2)

#Spline_3
matriz_Puntos_3 = np.array([[1.6, 4.71],[1.74, 4.52],[1.81, 4.3], [1.88, 4.13], [2.17, 4.19],[2.19,4.08],[1.96,4.07],[1.79,3.91],[1.69,3.92],[1.69,4.27],[1.61,4.54],[1.6,4.71]])
colores_3 = 'brown'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_4
matriz_Puntos_4 = np.array([[2.17, 4.19],[2.74, 4.07],[3.31, 3.67], [3.68, 3.03], [3.48, 2.39],[2.86,1.8],[2.35,2.3],[1.72,2.4],[1.48,2.94],[1.32,2.91],[1.69,3.92],[1.79,3.91],[1.96,4.07],[2.19,4.08],[2.17,4.19]])
colores_4 = 'limegreen'
array_matriz_puntos.append(matriz_Puntos_4)
colores_cada_spline.append(colores_4)

#Spline_5
matriz_Puntos_5 = np.array([[3.61, 2.61],[3.71, 2.26],[3.21, 1.68], [2.25, 1.47], [1.39, 1.89],[1.06,2.39],[1.32,2.91],[1.48,2.94],[1.72,2.4],[2.35,2.3],[2.86,1.8],[3.48,2.39],[3.61,2.61]])
colores_5 = 'gold'
array_matriz_puntos.append(matriz_Puntos_5)
colores_cada_spline.append(colores_5)

#Spline_6
matriz_Puntos_3 = np.array([[1.92, 3.51],[2.19, 2.99],[1.93, 2.48], [1.75, 2.97], [1.92, 3.51]])
colores_3 = 'gold'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_7
matriz_Puntos_3 = np.array([[1.96, 3.36],[2.07, 3.01],[1.96, 2.67], [1.85, 3.05], [1.96, 3.36]])
colores_3 = 'white' 
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)


#Spline_8
matriz_Puntos_3 = np.array([[3.21, 1.68],[3.75, 1.22],[3.37, 0.82], [2.5, 1], [2.21, 0.84],[2.1,0.4],[1.66,0.42],[1.54,0.73],[1.13,0.82],[0.79,0.5],[0.37,0.78],[0.41,1.28],[0.38,1.46],[0.5,1.5],[0.9,1.71],[1.23,2.02],[1.39,1.89],[2.25,1.47],[3.21,1.68]])
colores_3 = 'limegreen'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_9
matriz_Puntos_3 = np.array([[3.29, 1.59],[3.75, 1.22],[3.37, 0.82], [2.89, 0.89], [2.83, 0.98],[3.04,1.1],[2.96,1.39],[3.29,1.38],[3.29,1.59]])
colores_3 = 'gold'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_10
matriz_Puntos_3 = np.array([[2.21, 0.68],[2.1, 0.4],[1.66, 0.42], [1.54, 0.73], [1.68, 0.6],[1.88,0.69],[2.05,0.57],[2.21,0.68]])
colores_3 = 'gold'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_11
matriz_Puntos_3 = np.array([[1.02, 0.76],[0.87, 0.71],[0.71, 0.96], [0.47, 0.85], [0.37, 1.01],[0.37,0.78],[0.78,0.5],[1.02,0.76]])
colores_3 = 'gold'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)


#Spline_12
matriz_Puntos_3 = np.array([[1.65, 1.45],[1.67, 1.04],[1.13, 0.82], [0.84, 1.21], [0.99, 1.57],[1.08,1.52],[1.32,1.64],[1.51,1.43],[1.65,1.45]])
colores_3 = 'gold'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_13
matriz_Puntos_3 = np.array([[1.12, 2.61],[1.06, 2.39],[1.23, 2.02], [0.9, 1.71], [0.5, 1.5],[0.49,1.69],[0.59,1.79],[0.71,2.24],[1.12,2.61]])
colores_3 = 'brown'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)


#Spline_14
matriz_Puntos_3 = np.array([[1.07, 2.49],[1.1, 2.24],[0.86, 1.97], [0.92, 2.27], [1.07, 2.49]])
colores_3 = 'brown'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_15
matriz_Puntos_3 = np.array([[0.64, 2.04],[0.68, 1.87],[0.59, 1.69], [0.64, 2.04]])
colores_3 = 'brown'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#######################################################################################################################3333
#######################################################################################################################3333
#######################################################################################################################3333
#######################################################################################################################3333
#Spline_1
matriz_Puntos_3 = np.array([[5.3, 5.84],[5.51, 5.61],[5.49, 5.42], [5.68, 5.2],[5.65,5.04],[5.94,4.8],[6.11,4.5],[6.04,4.25],[5.87,4.62],[5.57,4.89],[5.26,4.67],[5.32,4.94],[4.94,4.55],[4.73,4.79],[4.68,4.56],[4.55,4.71],[4.3,4.32],[4.5,4.75],[5.03,5.07],[4.96,5.27],[5.28,5.51],[5.3,5.89]])
colores_3 = 'orange'
array_matriz_puntos.append(matriz_Puntos_3)
colores_cada_spline.append(colores_3)

#Spline_2
matriz_Puntos = np.array([[6.04, 4.25],[6.11, 4.5],[6.35, 4.75], [6.54, 4.2],[6.38,3.68],[5.97,3.44],[5.38,2.98],[4.49,3.31],[4.47,3.51],[4.36,3.82],[4.3,4.04],[4.3,4.32]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_3
matriz_Puntos = np.array([[4.44, 4.21],[4.52, 4.3],[4.63, 4.25],[4.77, 3.98],[4.8, 3.55], [4.61, 3.48],[4.42,3.9],[4.44,4.21]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)


#Spline_4
matriz_Puntos = np.array([[4.53, 3.64],[4.42, 3.9],[4.44, 4.21],[4.52, 4.3],[4.63, 4.25], [4.72, 4.1],[4.56,4.16],[4.48,3.93],[4.53,3.64]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_5
matriz_Puntos = np.array([[4.76, 3.87],[4.75, 3.58],[4.57, 3.82],[4.62, 4.08],[4.76, 3.87]])
colores = 'white'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_6
matriz_Puntos = np.array([[5.68, 4.5],[5.83, 4.36],[5.87, 4.07],[5.78, 3.68],[5.5, 3.65],[5.42,4.11],[5.51,4.41],[5.68,4.5]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_7
matriz_Puntos = np.array([[5.87, 4.07],[5.84, 3.92],[5.8,4.2],[5.62,4.36],[5.42,4.11],[5.51,4.41]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_8
matriz_Puntos = np.array([[5.57, 4.23],[5.7, 3.99],[5.59,3.69],[5.47,3.98],[5.57,4.23]])
colores = 'white'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_9
matriz_Puntos = np.array([[5.69, 3.58],[5.61, 3.23],[5.38,3.05],[5.12,3.11],[4.83,3.36],[5.22,3.38],[5.69,3.58]])
colores = 'darkred'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_10
matriz_Puntos = np.array([[5.61, 3.23],[5.32, 3.27],[5.12,3.11]])
colores = 'red'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_11
matriz_Puntos = np.array([[4.97, 3.36],[4.93, 3.28],[4.83,3.36]])
colores = 'gray'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_12
matriz_Puntos = np.array([[5.69, 3.58],[5.64, 3.43],[5.57,3.51]])
colores = 'gray'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_13
matriz_Puntos = np.array([[6.23, 4.36],[6.34, 4.1],[6.28,3.76],[6.04,3.59],[6.07,3.96],[6.23,4.36]])
colores = 'coral'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_14
matriz_Puntos = np.array([[4.21, 4.09],[4.3, 4.04],[4.36,3.82],[4.44,3.53],[4.26,3.71],[4.21,4.09]])
colores = 'coral'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_15
matriz_Puntos = np.array([[4.3, 4.32],[4.16,4.38],[4.08,4.09],[4.12,3.76],[4.43,3.40]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_16
matriz_Puntos = np.array([[5.47, 3],[5.58,2.93],[6.6,3.11],[6.65,3.01],[6.6,2.92],[5.62,2.77],[5.92,2.38],[6.15,2.3],[6.03,2.25],[5.91,2.28],[5.77,2.22],[5.92,2.38]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_17
matriz_Puntos = np.array([[6.65, 3.11],[7.03,3.35],[7.07,3.32],[7.14,3.33],[7.26,3.15],[7.49,2.95],[7.01,2.86],[7.19,2.64]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_18
matriz_Puntos = np.array([[7.05,2.55],[6.88,2.72],[6.76,2.75],[6.66,2.92]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)


#Spline_19
matriz_Puntos = np.array([[5.34, 2.97],[5.32,2.89],[4.91,2.71],[4.34,2.41],[4.31,2.29],[4.4,2.24],[4.68,2.42],[5.32,2.72]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)


#Spline_20
matriz_Puntos = np.array([[4.34, 2.41],[3.86,2.48],[3.72,2.38],[3.71,2.26],[3.64,2.05],[3.9,2.13],[3.71,1.88],[3.77,1.76],[4.12,2.03],[4.24,1.73],[4.37,1.82],[4.34,2.05],[4.4,2.24]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_21
matriz_Puntos = np.array([[5.77,2.22],[5.91,2.28],[6.03,2.25],[6.15, 2.3],[6.32,2.2],[6.46,1.87],[6.53,1.4],[6.38,1.24],[6.19,1.33],[6.07,1.59],[5.85,1.8],[5.79,1.93],[5.6,2.07],[5.77,2.22],[5.64,2.34]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_22
matriz_Puntos = np.array([[5.52,2.53],[5.47,2.56],[5.4,2.4],[5.46, 2.26],[5.57,2.23],[5.64,2.34],[5.63,2.6],[5.46,2.76],[5.33,2.62],[5.27,2.23],[5.3,1.83],[5.54,1.56],[5.79,1.53],[5.95,1.6]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_23
matriz_Puntos = np.array([[6.07,1.59],[5.95,1.6],[5.85,1.8]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_24
matriz_Puntos = np.array([[5.33,2.62],[5.32,2.89],[5.27,2.23],[5.18,1.7],[5.29,1.39],[5.58,1.11],[5.7,1.01],[5.84,1.1],[5.77,1.32],[5.79,1.53],[5.95,1.6]])
colores = 'orange'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_25
matriz_Puntos = np.array([[5.58,1.11],[5.69,0.56],[5.75,0.3],[5.87,0.33],[6,0.32],[6.06,0.38],[6.22,0.37],[6.25,0.52],[6.03,0.89],[5.88,1.18],[5.79,1.25]])
colores = 'burlywood'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_26
matriz_Puntos = np.array([[5.8,1.46],[5.97,1.44],[6.12,1.5]])
colores = 'red'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_27
matriz_Puntos = np.array([[6.51,1.88],[6.53,2.08],[6.61,2.32],[6.53,2.3],[6.56,2.73],[6.42,2.57],[6.47,2.91]])
colores = 'red'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_28
matriz_Puntos = np.array([[6.22,2.88],[6,2.6],[5.97,2.75],[5.85,2.48]])
colores = 'red'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_29
matriz_Puntos = np.array([[6,2.37],[6.09,2.45],[6.22,2.46],[6.13,2.6],[6.26,2.68],[6.31,2.52],[6.27,2.64],[6.4,2.2],[6.47,2.32],[6.44,2.01]])
colores = 'yellow'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#######################################################################################################################3333
#######################################################################################################################3333
#######################################################################################################################3333
#######################################################################################################################3333
#Spline_1
matriz_Puntos = np.array([[8.05,2.69],[7.83,2.32],[8,2],[8.34,2.11],[8.55,2.35],[8.87,2.19],[9.28,2.23],[9.59,2.61],[9.48,2.97],[9.72,3.28],[9.93,3.94],[9.81,4.54],[9.32,5.1],[8.3,5.3],[7.69,5.08],[7.26,4.64],[7.25,4.48],[7.4,4.38],[7.76,4.78],[7.9,4.55],[8.25,4.67]])
colores = 'darkblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_2
matriz_Puntos = np.array([[7.34,4.05],[7.25,4.48],[7.4,4.38],[7.76,4.78],[7.9,4.55],[8.25,4.67],[8.04,4.27],[7.97,3.81]])
colores = 'lightblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_3
matriz_Puntos = np.array([[7.97,3.81],[7.53,4.21],[7.34,4.05],[7.3,3.58],[7.71,3.25],[7.97,3.81]])
colores = 'yellow'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_3
matriz_Puntos = np.array([[7.3,3.58],[7.65,3.61],[7.97,3.81]])
colores = 'yellow'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_4
matriz_Puntos = np.array([[7.26,4.64],[7.04,4.03],[7.04,3.58],[7.14,3.33],[7.26,3.15],[7.49,2.95],[8.05,2.69]])
colores = 'darkblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_5
matriz_Puntos = np.array([[7.04,4.03],[7.14,3.97],[7.22,3.68],[7.24,3.3],[7.14,3.33],[7.04,3.58],[7.04,4.03]])
colores = 'white'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_6
matriz_Puntos = np.array([[8.5,4.08],[8.72,4.53],[9.01,4.16],[8.83,3.7],[8.5,4.08],[8.72,4.53]])
colores = 'white'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_7
matriz_Puntos = np.array([[8.05,2.69],[9.04,2.89],[9.49,3.49],[9.54,4.29],[9.17,4.86],[8.64,5.01],[8.25,4.67]])
colores = 'darkblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_8
matriz_Puntos = np.array([[9.72,3.28],[10.27,3.5],[10.92,3.6],[10.4,2.86],[9.75,2.44]])
colores = 'lightblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_9
matriz_Puntos = np.array([[9.92,2.52],[10.32,1.74],[10.24,1.19],[9.84,0.95],[9.22,1],[8.67,0.85]])
colores = 'lightblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_10
matriz_Puntos = np.array([[8.05,2.69],[7.19,2.64],[7.059,2.55],[7.43,2.2],[8.14,1.94],[8.16,1.53]])
colores = 'lightblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_11
matriz_Puntos = np.array([[8.16,1.53],[7.64,1.68],[7.88,1],[8.61,0.59],[8.67,0.85],[8.16,1.53]])
colores = 'yellow'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_12
matriz_Puntos = np.array([[10.24,1.19],[10.4,0.81],[10.38,0.34],[9.94,0.62],[9.84,0.95],[10.24,1.19]])
colores = 'yellow'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_13
matriz_Puntos = np.array([[10.24,2.71],[10.37,2.66],[10.52,2.85],[10.56,2.63],[10.93,3.02],[10.93,2.71],[10.72,2.38],[10.67,2],[10.32,1.74]])
colores = 'darkblue'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_14
matriz_Puntos = np.array([[8.29,2.04],[8.49,1.8],[8.43,1.46],[8.2,1.75],[8.29,2.04]])
colores = 'gray'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

#Spline_15
matriz_Puntos = np.array([[9.27,2.2],[9.4,1.95],[9.32,1.59],[9.06,1.85],[9.08,2.16],[9.27,2.2]])
colores = 'gray'
array_matriz_puntos.append(matriz_Puntos)
colores_cada_spline.append(colores)

In [ ]:
# Dibujar ambas funciones en el mismo gráfico
Dibujar_Completo(array_matriz_puntos, colores_cada_spline,'k')

In [ ]:
# La funcion "plotspline" que nos pide el ejercicio dibuja un spline, pasandole
# como parametros los puntos por lo que se pretende que pase el spline.

# Por otra parte, esta hace uso de "spline" (la segunda funcion que nos pide el ejercicio), 
# mediante la cual se calcula la lista de polinomios que constituye el spline.